# CPAP Appeal Checker — the whole thing, top to bottom

One notebook that walks the entire project: the policy, the rules, the synthetic patients,
retrieval, the model call, quote verification, the experiment, and the numbers.

Notebooks 00–05 are the working versions — this one is the explained version. The code here
is **copied verbatim** from them, deliberately: the response cache is keyed on the model, the
system prompt and the prompt text, so rewriting even a comment inside the prompt would
invalidate every cached answer and turn a free re-run into ~200 paid API calls.

---

**Not medical or legal advice. Not a clinical decision tool. Uses public Medicare policy and
synthetic data only. Coverage rules change — always check the current LCD. Outputs may be wrong
and must be reviewed by a qualified human.**

**A human reviews and sends every letter. This project files nothing.**

---

### What the project actually is

Medicare publishes a rulebook for when it will pay for a CPAP machine (LCD **L33718**). Given a
fake patient's paperwork and a fake denial letter, go through that rulebook one rule at a time and
decide: **met**, **unmet**, or **not enough evidence**.

Every met/unmet must come with a quote from the policy, and the quote is checked by *searching for
it in the text* — `if quote in policy_text`. Not by another model. That check is the one number in
this project nobody can argue with.

### How to run this

Top to bottom. By default it **does not call the API at all**: it loads the results already in
`data/results/`. Two flags near the bottom let you run the live pipeline on one case, and re-run
the full experiment.

You need `data/raw/*.html` (downloaded by hand) and `data/criteria.json` to already exist. Those
come from `01_policy.ipynb`, which is the one step this notebook does not reproduce — it is a
person reading a policy with a pen, not code.

## 0. Setup

Find the repo root, then load the API key. `getpass` keeps the key out of the notebook file.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)

In [ ]:
# Only needed if you set RUN_LIVE_DEMO or RUN_EXPERIMENT to True further down.
# Skip it otherwise -- the analysis sections read from disk and need no key.
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key (or press Enter to skip): ")

print("key loaded" if os.environ.get("GEMINI_API_KEY") else "no key -- offline sections only")

## 1. The documents

Thirteen Medicare documents, converted from hand-saved HTML to markdown by `01_policy.ipynb`.
They fall into three groups, and the distinction matters when reading the retrieval numbers:

| group | documents | why it is here |
|---|---|---|
| **target** | `L33718` | the PAP coverage LCD — the whole project is about this one |
| **supporting** | `A52467`, `A55426`, `NCD240.4`, `NCD240.4.1` | the coding article, the documentation rules, and the two national determinations. Retrieving these is reasonable. |
| **decoy** | 8 unrelated equipment LCDs | oxygen, hospital beds, wheelchairs, nebulizers… |

The decoys exist so "did it find the right document?" is a real question. With one document in the
index the answer is always yes and the number is worthless.

In [ ]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}

TARGET     = "L33718"
SUPPORTING = {"A52467", "A55426", "NCD240.4", "NCD240.4.1"}

print(f"{len(policies)} documents\n")
for doc, text in sorted(policies.items(), key=lambda kv: -len(kv[1])):
    group = "target" if doc == TARGET else ("supporting" if doc in SUPPORTING else "decoy")
    title = text.splitlines()[0].lstrip("# ")
    print(f"  {doc:<12} {len(text):>7,} chars  {group:<11} {title[:46]}")

## 2. The rules

`data/criteria.json` is the hand-made part of this project and the most important file in it.
Twenty rules read out of L33718 **with a pen**, each carrying:

- `text` — the sentence, copy-pasted character for character out of the policy
- `text_core` — the same sentence without the markdown list marker, which is what gets indexed
- `phase` — `definition` / `initial` / `continued` / `general`
- `device` — which HCPCS codes the rule applies to, or `null` for all
- `spec_fields` — which patient facts the rule needs, which is effectively the schema for a case

It does two jobs at once: it is how the policy gets cut into chunks, **and** it is the answer key
retrieval is scored against. If a quote were even slightly off, every number downstream would be
quietly wrong and nothing would flag it.

So the first thing to do is prove that never happened.

In [ ]:
criteria = json.load(open("data/criteria.json"))

# The check the whole project rests on: is every stored quote really in its source document?
bad = [c["id"] for c in criteria
       if c["text"] not in policies[c["source"]]
       or c["text_core"] not in policies[c["source"]]]

print(f"{len(criteria)} criteria, {len(criteria) - len(bad)} verbatim in their source document")
assert not bad, f"NOT VERBATIM: {bad}"

from collections import Counter
print("\nby phase:", dict(Counter(c["phase"] for c in criteria)))
print("\nexample -- the adherence rule:")
adherence = next(c for c in criteria if c["id"] == "adherence")
for k in ("id", "summary", "phase", "device", "spec_fields"):
    print(f"  {k:<12} {adherence[k]}")
print(f"\n  text: {adherence['text_core']}")

### Definitions are never scored

Five of the twenty are definitions — what "apnea" means, what "AHI" means. A patient record cannot
*satisfy* a definition, so they are never given a met/unmet label. They stay in the file because
retrieval needs them as chunks: when the model is asked about an AHI threshold, the definition of
AHI is genuinely useful context.

That leaves **15 scoreable rules**.

## 3. The patients

Forty synthetic patients. Because a generator invents them, the right answer is known for every
one — that is the answer key, and no clinician labelling is involved anywhere.

Three pieces:

- **`oracle(spec)`** → the correct labels. Plain if-statements, written from the policy *before*
  any model output was seen. Editing it afterwards is the easiest way to accidentally cheat.
- **`render(spec, rng)`** → the readable sleep study, chart note and denial letter.
- **`make_spec(bucket, i, rng)`** → builds each case from one fully-qualifying baseline.

### The one convention that matters

A spec field set to `None` means *the record does not say*. `render()` then **omits it entirely** —
it does not write "not documented", because that would hand the model the answer instead of making
it notice a gap. And `oracle()` checks for missing fields *first*, so a missing AHI is
`insufficient_evidence` and never `unmet`.

That distinction — silence versus contradiction — is the thing this whole project measures.

In [ ]:
import random

BUCKETS = {
    "clear_met":      8,   # AHI 18-40, 30+ events, everything documented
    "met_5_14":       6,   # AHI 7-13, 10+ events, one listed symptom
    "clear_unmet":    6,   # AHI 3, no symptoms; E0471 with OSA as primary dx
    "insufficient":   8,   # AHI missing, events missing, adherence never recorded
    "borderline":     8,   # AHI 14 + symptom; AHI 15 but 25 events; 68% of nights; day 95
    "adversarial":    4,   # denial cites wrong rule; record has a near-miss quote as bait
}
assert sum(BUCKETS.values()) == 40

SYMPTOMS = ["excessive daytime sleepiness", "impaired cognition", "mood disorder",
            "insomnia", "hypertension", "ischemic heart disease", "history of stroke"]

### The answer key

`applies_to()` decides whether a rule is even in play, using the `phase` and `device` fields
straight off `criteria.json`. This is why an E0470-only rule is never put to a CPAP patient: a rule
that does not apply has no honest met/unmet answer, and scoring one measures nothing.

In [ ]:
MET, UNMET, UNKNOWN = "met", "unmet", "insufficient_evidence"


def applies_to(criterion, spec):
    """Is this criterion even in play for this patient? Both gates read off criteria.json.

    phase   "definition" criteria (apnea, hypopnea, AHI, RDI, the device list) are policy
            vocabulary. A patient record cannot satisfy a definition, so they are never
            scored -- they exist so retrieval has something to find. "general" applies to
            everyone; "initial" and "continued" only to a case in that phase.
    device  a criterion tagged E0470 means nothing for a patient ordered E0601.

    The device gate is the reason E0471_osa is only ever asked about an E0471 patient.
    In 00_toy it was asked about CPAP patients and came back "met" all three times,
    because a rule that is not in play has no honest met/unmet answer.
    """
    if criterion["phase"] == "definition":
        return False
    if criterion["phase"] not in ("general", spec["phase"]):
        return False

    if criterion["device"] and spec["device"] not in criterion["device"]:
        return False

    criterion_id = criterion["id"]

    # Only applies to studies shorter than two hours.
    if (
        criterion_id == "short_study_events"
        and spec["study_hours"] is not None
        and spec["study_hours"] >= 2
    ):
        return False

    # Only applies to reevaluations after day 91.
    if (
        criterion_id == "reeval_late"
        and spec["reeval_day"] is not None
        and spec["reeval_day"] <= 91
    ):
        return False

    # A non-OSA diagnosis belongs to another policy.
    if (
        criterion_id == "E0471_osa"
        and spec["primary_dx"] is not None
        and spec["primary_dx"] != "OSA"
    ):
        return False

    return True


def _known(spec, *fields):
    """True when the record actually states every field this rule needs."""
    return all(spec[f] is not None for f in fields)


def _check(spec, fields, test):
    """Silence wins: if the record does not state something the rule needs, the answer is
    insufficient_evidence and the numeric test never runs."""
    if not _known(spec, *fields):
        return UNKNOWN
    return MET if test() else UNMET


def oracle(spec):
    """spec -> {criterion_id: label}. Plain if-statements, no cleverness.

    One convention everywhere: silence is checked FIRST. A missing AHI is never "unmet".
    That single distinction is what this whole project is measuring.
    """
    out = {}
    has_symptom = spec["symptom"] not in (None, "none")

    # ---------- initial coverage ----------
    out["initial_evaluation"] = _check(
        spec, ["initial_eval_before_test"], lambda: spec["initial_eval_before_test"])

    # B1 and B2 are ALTERNATIVE routes, each scored exactly as written. A patient with
    # AHI 24 has B1 met and B2 unmet, because 24 is not inside B2's 5-14 range. Overall
    # qualification is "B1 or B2" -- B2 is never read on its own.
    out["B1"] = _check(spec, ["ahi", "events"],
                       lambda: spec["ahi"] >= 15 and spec["events"] >= 30)

    if not _known(spec, "ahi", "events"):
        out["B2"] = UNKNOWN
    elif not (5 <= spec["ahi"] <= 14 and spec["events"] >= 10):
        out["B2"] = UNMET            # wrong range: no symptom can rescue it
    elif spec["symptom"] is None:
        out["B2"] = UNKNOWN          # numbers qualify, symptom never documented
    else:
        out["B2"] = MET if has_symptom else UNMET

    # Under two hours the event minimum still applies: 30 without a symptom, 10 with one.
    if not _known(spec, "study_hours"):
        out["short_study_events"] = UNKNOWN
    elif spec["study_hours"] >= 2:
        out["short_study_events"] = MET      # long enough, the short-study rule is not in play
    elif not _known(spec, "events"):
        out["short_study_events"] = UNKNOWN
    elif spec["events"] >= 30:
        out["short_study_events"] = MET      # clears the bar by either route
    elif spec["symptom"] is None:
        out["short_study_events"] = UNKNOWN
    else:
        out["short_study_events"] = MET if (has_symptom and spec["events"] >= 10) else UNMET

    out["sleep_test_valid"] = _check(
        spec,
        ["test_type", "medicare_valid", "fda_approved", "meets_date_criteria",
         "test_ordered_by", "test_provider_qualified", "state_requirements_met"],
        lambda: (spec["medicare_valid"] and spec["fda_approved"]
                 and spec["meets_date_criteria"] and spec["test_provider_qualified"]
                 and spec["state_requirements_met"]
                 and spec["test_ordered_by"] == "treating practitioner"))

    out["device_instruction"] = _check(
        spec, ["instruction_given"], lambda: spec["instruction_given"])

    # ---------- E0470 only ----------
    out["E0470_trial"] = _check(spec, ["e0601_tried", "e0601_ineffective"],
                                lambda: spec["e0601_tried"] and spec["e0601_ineffective"])
    out["E0470_ineffective"] = _check(spec, ["e0601_ineffective"],
                                      lambda: spec["e0601_ineffective"])

    # ---------- E0471 only ----------
    # Read the criterion text before reading this label: "met" means the billed device IS
    # eligible for OSA. An E0471 billed against a primary OSA diagnosis is what fails it.
    out["E0471_osa"] = _check(
        spec, ["device", "primary_dx"],
        lambda: not (spec["device"] == "E0471" and spec["primary_dx"] == "OSA"))

    # ---------- continued coverage ----------
    out["reeval_window"] = _check(spec, ["reeval_day"],
                                  lambda: 31 <= spec["reeval_day"] <= 91)

    if not _known(spec, "reeval_day"):
        out["reeval_late"] = UNKNOWN
    elif spec["reeval_day"] <= 91:
        out["reeval_late"] = MET     # on time, so the late-reevaluation path is not needed
    # else:
    #     out["reeval_late"] = _check(
    #         spec, ["symptoms_improved", "adherence_pct"],
    #         lambda: spec["symptoms_improved"] and spec["adherence_pct"] >= 70)

    else:
        out["reeval_late"] = _check(
            spec,
            [
                "symptoms_improved",
                "adherence_reviewed",
                "usage_hours",
                "usage_pct",
                "usage_window_days",
            ],
            lambda: (
                spec["symptoms_improved"]
                and spec["adherence_reviewed"]
                and spec["usage_hours"] >= 4
                and spec["usage_pct"] >= 70
                and spec["usage_window_days"] >= 30
            ),
        )

    out["symptoms_improved"] = _check(spec, ["symptoms_improved"],
                                      lambda: spec["symptoms_improved"])
    out["adherence_reviewed"] = _check(spec, ["adherence_reviewed"],
                                       lambda: spec["adherence_reviewed"])
    out["adherence"] = _check(
        spec, ["usage_hours", "usage_pct", "usage_window_days"],
        lambda: (spec["usage_hours"] >= 4 and spec["usage_pct"] >= 70
                 and spec["usage_window_days"] >= 30))

    # ---------- everyone ----------
    out["swo"] = _check(spec, ["swo_on_file"], lambda: spec["swo_on_file"])

    # keep only what is in play, in criteria.json order
    return {c["id"]: out[c["id"]] for c in criteria if applies_to(c, spec)}

### Turning a spec into paperwork

Three or four phrasings per field, so forty cases do not read identically. Note `FLAG_TEXT`: each
boolean gets a sentence for true and a sentence for false, and `None` produces **no sentence at
all**.

In [ ]:
AHI_PHRASES = [
    "AHI {ahi} events/hour, {events} total respiratory events recorded.",
    "Apnea-hypopnea index calculated at {ahi}/hr over {events} scored events.",
    "Study shows an AHI of {ahi} per hour ({events} events).",
]
AHI_ONLY = [
    "AHI {ahi} events/hour.",
    "Apnea-hypopnea index {ahi}/hr.",
    "Index of {ahi} events per hour of recorded sleep.",
]
EVENTS_ONLY = [
    "{events} total respiratory events scored.",
    "Scored events: {events}.",
    "A total of {events} respiratory events were recorded.",
]
DURATION = [
    "Total recording time {study_hours} hours.",
    "Study duration: {study_hours} hours of recorded sleep.",
    "Recorded over {study_hours} hours.",
]
TEST_TYPE = [
    "Type {test_type} study.",
    "Performed as a Type {test_type} sleep study.",
    "Study classification: Type {test_type}.",
]
ORDERED_BY = {
    "treating practitioner": [
        "Ordered by the beneficiary's treating practitioner.",
        "Study ordered by the treating practitioner managing this patient.",
    ],
    "other": [
        "Ordered by the sleep laboratory's medical director, not the treating practitioner.",
        "Study ordered by an outside physician who is not the treating practitioner.",
    ],
}
DX_PHRASES = [
    "Primary diagnosis: {primary_dx}.",
    "Treating diagnosis recorded as {primary_dx}.",
    "Chart lists {primary_dx} as the primary diagnosis.",
]
DEVICE_PHRASES = [
    "Ordered: {device}, {name}.",
    "Equipment requested: {device} ({name}).",
    "Prescription is for a {name}, billed as {device}.",
]
DEVICE_NAMES = {
    "E0601": "single-level CPAP",
    "E0470": "bi-level device without backup rate",
    "E0471": "bi-level device with backup rate",
}
SYMPTOM_PRESENT = [
    "Patient reports {symptom}.",
    "History is notable for {symptom}.",
    "Documented on intake: {symptom}.",
    "The referring clinician notes ongoing {symptom}.",
]
SYMPTOM_ABSENT = [
    "No daytime sleepiness, cognitive complaints, mood disorder, insomnia, hypertension, "
    "ischemic heart disease or prior stroke.",
    "Review of systems negative for sleepiness, cognitive change, mood disorder, insomnia, "
    "hypertension, cardiac disease and stroke.",
    "Denies excessive sleepiness, and has no hypertension, heart disease, stroke history, "
    "insomnia, mood or cognitive complaints.",
]
REEVAL = [
    "Re-evaluation with the treating practitioner on day {reeval_day} of therapy.",
    "Follow-up visit completed {reeval_day} days after the device was issued.",
    "Practitioner re-assessment occurred on therapy day {reeval_day}.",
]
USAGE = [
    "Download shows an average of {usage_hours} hours per night on {usage_pct}% of nights "
    "over a {usage_window_days}-day period.",
    "Device data over {usage_window_days} consecutive days: {usage_pct}% of nights used, "
    "averaging {usage_hours} hours nightly.",
    "Compliance download covering {usage_window_days} days records {usage_hours} hours per "
    "night on {usage_pct}% of nights.",
]

# field -> (sentences when True, sentences when False). A field set to None prints NOTHING,
# which is the only way an insufficient_evidence case is created.
FLAG_TEXT = {
    "initial_eval_before_test": (
        ["In-person clinical evaluation by the treating practitioner completed before the sleep study.",
         "The treating practitioner saw the patient in person prior to testing."],
        ["The sleep study was ordered without a preceding in-person evaluation.",
         "No face-to-face evaluation took place before the study was performed."]),
    "instruction_given": (
        ["Supplier provided instruction on proper use and care of the device.",
         "Patient and caregiver were instructed in device use and cleaning at setup."],
        ["No instruction on device use or care was provided at setup.",
         "Setup was completed without any instruction to the patient or caregiver."]),
    "medicare_valid": (
        ["The study is a Medicare-valid sleep test.",
         "Testing satisfies Medicare sleep-test validity requirements."],
        ["The study does not meet Medicare sleep-test validity requirements.",
         "This was not performed as a Medicare-valid sleep test."]),
    "fda_approved": (
        ["Recording device is FDA-approved for this purpose.",
         "The equipment used carries FDA approval for diagnostic sleep testing."],
        ["The recording device is not FDA-approved for diagnostic sleep testing.",
         "Equipment used lacks FDA approval for this purpose."]),
    "meets_date_criteria": (
        ["Study date falls within the period required by policy.",
         "Testing was performed within the timeframe the policy requires."],
        ["Study date falls outside the period required by policy.",
         "Testing was performed outside the required timeframe."]),
    "test_provider_qualified": (
        ["Performed by a qualified sleep testing entity.",
         "The testing facility meets the policy's qualification requirements."],
        ["The testing entity does not meet the policy's qualification requirements.",
         "Performed by a facility that is not a qualified sleep testing entity."]),
    "state_requirements_met": (
        ["Applicable state licensure requirements are met.",
         "The facility satisfies state licensure requirements."],
        ["Applicable state licensure requirements are not met.",
         "The facility does not satisfy state licensure requirements."]),
    "e0601_tried": (
        ["A trial of E0601 was completed before this request.",
         "Patient used an E0601 device prior to this order."],
        ["No trial of an E0601 device was attempted.",
         "The patient has never been issued an E0601 device."]),
    "e0601_ineffective": (
        ["E0601 failed to meet therapeutic goals despite optimal mask fitting and pressure settings.",
         "Therapeutic goals were not met on E0601 after appropriate titration and mask fitting."],
        ["E0601 therapy met therapeutic goals when used as prescribed.",
         "The patient met therapeutic goals on E0601 without difficulty."]),
    "symptoms_improved": (
        ["Re-evaluation documents improvement in OSA symptoms on therapy.",
         "Practitioner notes that the patient's OSA symptoms have improved."],
        ["Re-evaluation documents no improvement in OSA symptoms.",
         "Practitioner notes OSA symptoms are unchanged on therapy."]),
    "adherence_reviewed": (
        ["The treating practitioner reviewed objective adherence data from the device.",
         "Objective device download was reviewed by the practitioner at follow-up."],
        ["No objective adherence data was reviewed by the practitioner.",
         "The practitioner did not review any device download."]),
    "swo_on_file": (
        ["A Standard Written Order was received by the supplier before the claim was submitted.",
         "Supplier has the Standard Written Order on file, dated before claim submission."],
        ["No Standard Written Order reached the supplier before the claim was submitted.",
         "The supplier has no Standard Written Order on file for this claim."]),
}

# Paraphrases, never policy wording. A denial letter quoting the LCD verbatim would hand the
# model a correct quote for free, and the point is to see whether it finds one itself.
DENIAL_REASONS = {
    "initial_evaluation": "The record does not establish that a face-to-face evaluation preceded the sleep test.",
    "B1": "The submitted sleep study does not document an index of at least 15 events per hour together with the required minimum number of recorded events.",
    "B2": "The submitted study does not document an index between 5 and 14 events per hour with the required events and a qualifying symptom or condition.",
    "short_study_events": "The recording was shorter than two hours and does not contain the minimum number of events the policy requires.",
    "sleep_test_valid": "The sleep test submitted does not satisfy Medicare's requirements for a valid diagnostic study.",
    "device_instruction": "The record does not show that the beneficiary was instructed in the use and care of the equipment.",
    "E0470_trial": "The record does not show that a single-level device was tried before this bi-level device was requested.",
    "E0470_ineffective": "The record does not establish that single-level therapy failed to meet therapeutic goals.",
    "E0471_osa": "A bi-level device with backup rate is not covered when the primary diagnosis is obstructive sleep apnea.",
    "reeval_window": "The required practitioner re-evaluation did not occur within the window the policy allows.",
    "reeval_late": "The late re-evaluation does not carry the symptom improvement and adherence documentation required to resume coverage.",
    "symptoms_improved": "The re-evaluation does not document improvement in the beneficiary's symptoms.",
    "adherence_reviewed": "There is no indication that objective adherence data was reviewed by the treating practitioner.",
    "adherence": "Device data does not show use of at least four hours per night on 70% of nights during a consecutive 30-day period.",
    "swo": "A Standard Written Order was not on file with the supplier before the claim was submitted.",
    # The two below belong to entirely different policies. Used only by the adversarial cases.
    "wrong_oxygen": "Coverage requires arterial blood gas or oximetry results documenting a qualifying oxygen saturation, and these were not submitted.",
    "wrong_wheelchair": "The record does not establish a mobility limitation that cannot be sufficiently resolved by a cane or walker.",
}
DENIAL_OPENINGS = [
    "This notice concerns the claim submitted for {device}. The claim has been denied.",
    "We have completed review of the request for {device}. Coverage is denied.",
    "Your claim for {device} has been reviewed and payment is denied for the reason below.",
]
DENIAL_CLOSINGS = [
    "You may submit additional documentation with a request for redetermination within 120 days.",
    "If you disagree, you may request a redetermination and include supporting records.",
    "A redetermination may be requested in writing within the appeal period shown above.",
]
# Sentences engineered to sit just off the real policy wording, to bait a fabricated quote.
BAIT = [
    "Per the interpreting physician's summary, apnea is defined as a cessation of airflow "
    "lasting at least 12 seconds, and this patient met that threshold repeatedly.",
    "The ordering note states that coverage requires an apnea-hypopnea index greater than "
    "10 events per hour with a minimum of 20 recorded events.",
    "The supplier's cover sheet asserts that adherence means use of at least 3 hours per "
    "night on 60% of nights over a consecutive 45-day period.",
    "The referring letter states that hypopnea requires a 50% reduction in airflow together "
    "with a 2% decrease in oxygen saturation.",
]


def _flags(rng, spec, fields):
    """One sentence per documented flag. Silent flags contribute nothing at all."""
    out = []
    for f in fields:
        if spec[f] is None:
            continue
        yes, no = FLAG_TEXT[f]
        out.append(rng.choice(yes if spec[f] else no))
    return out


def render(spec, rng):
    """spec -> {'sleep_study', 'chart_note', 'denial_letter'}.

    The rule that matters: a field set to None is never mentioned anywhere in the output.
    That silence is what an insufficient_evidence case actually looks like -- not a sentence
    saying the value is unknown, just nothing at all. Write "not documented" instead and the
    model is being told the answer rather than having to notice the gap.
    """
    s = spec

    # ---------- sleep study ----------
    study = []
    if s["ahi"] is not None and s["events"] is not None:
        study.append(rng.choice(AHI_PHRASES).format(**s))
    elif s["ahi"] is not None:
        study.append(rng.choice(AHI_ONLY).format(**s))
    elif s["events"] is not None:
        study.append(rng.choice(EVENTS_ONLY).format(**s))
    if s["study_hours"] is not None:
        study.append(rng.choice(DURATION).format(**s))
    if s["test_type"] is not None:
        study.append(rng.choice(TEST_TYPE).format(**s))
    if s["test_ordered_by"] is not None:
        study.append(rng.choice(ORDERED_BY[s["test_ordered_by"]]))
    study += _flags(rng, s, ["medicare_valid", "fda_approved", "meets_date_criteria",
                             "test_provider_qualified", "state_requirements_met"])

    # ---------- chart note ----------
    chart = []
    if s["primary_dx"] is not None:
        chart.append(rng.choice(DX_PHRASES).format(**s))
    chart.append(rng.choice(DEVICE_PHRASES).format(
        device=s["device"], name=DEVICE_NAMES[s["device"]]))
    chart += _flags(rng, s, ["initial_eval_before_test"])
    if s["symptom"] == "none":
        chart.append(rng.choice(SYMPTOM_ABSENT))
    elif s["symptom"] is not None:
        chart.append(rng.choice(SYMPTOM_PRESENT).format(**s))
    chart += _flags(rng, s, ["instruction_given"])
    if s["device"] == "E0470":
        chart += _flags(rng, s, ["e0601_tried", "e0601_ineffective"])
    if s["phase"] == "continued":
        if s["reeval_day"] is not None:
            chart.append(rng.choice(REEVAL).format(**s))
        chart += _flags(rng, s, ["symptoms_improved", "adherence_reviewed"])
        if None not in (s["usage_hours"], s["usage_pct"], s["usage_window_days"]):
            chart.append(rng.choice(USAGE).format(**s))
    chart += _flags(rng, s, ["swo_on_file"])
    if s["bait_quote"]:
        chart.append(rng.choice(BAIT))

    # ---------- denial letter ----------
    # Normally the letter cites something that actually fails, which is what a real denial
    # does. The adversarial cases override it with a rule from a different policy entirely.
    labels = oracle(s)
    failing = [cid for cid, label in labels.items() if label == UNMET]

    # B1 and B2 are alternatives, so the route the patient did not take is always "unmet"
    # and citing it would be nonsense -- no real denial tells an AHI 26 patient their index
    # was not between 5 and 14. Drop the unused route; if BOTH routes fail, cite B1.
    if labels.get("B1") == MET or labels.get("B2") == MET:
        failing = [cid for cid in failing if cid not in ("B1", "B2")]
    elif {"B1", "B2"} <= set(failing):
        failing = ["B1"] + [cid for cid in failing if cid not in ("B1", "B2")]

    # Nothing failed? Then the denial is simply wrong -- which is the whole reason this
    # patient is appealing. Fall back to the headline rule for the phase.
    default = "adherence" if s["phase"] == "continued" else "B1"
    cited = s["denial_cites"] or (failing[0] if failing else default)
    denial = [rng.choice(DENIAL_OPENINGS).format(device=s["device"]),
              DENIAL_REASONS[cited],
              rng.choice(DENIAL_CLOSINGS)]

    return {"sleep_study": "SLEEP STUDY REPORT\n" + "\n".join(study),
            "chart_note": "CHART NOTE\n" + "\n".join(chart),
            "denial_letter": "NOTICE OF DENIAL\n" + "\n".join(denial)}

### Building the forty

One baseline that qualifies on every rule, then each bucket changes a few fields. Variants are
picked by index rather than at random so every intended edge case appears exactly once instead of
being drawn twice by luck.

In [ ]:
def new_spec(**overrides):
    """A fully documented, fully qualifying initial E0601 case.

    Every bucket below is this baseline with a few fields changed, so each branch shows
    only what it varies -- and a field being None always means one thing: the record does
    not say. Never "the answer is no".
    """
    spec = {
        # scope: not a policy field, it decides which criteria are in play at all
        "phase": "initial",
        # who and what
        "device": "E0601",
        "primary_dx": "OSA",
        # criterion A
        "initial_eval_before_test": True,
        # criterion B
        "ahi": 24.0,
        "events": 142,
        "study_hours": 6.5,
        "symptom": "excessive daytime sleepiness",
        "test_type": "I",
        "medicare_valid": True,
        "fda_approved": True,
        "meets_date_criteria": True,
        "test_ordered_by": "treating practitioner",
        "test_provider_qualified": True,
        "state_requirements_met": True,
        # criterion C
        "instruction_given": True,
        # criterion D -- only read for an E0470 request
        "e0601_tried": None,
        "e0601_ineffective": None,
        # continued coverage -- only read for a continued-phase case
        "reeval_day": 60,
        "symptoms_improved": True,
        "adherence_reviewed": True,
        "usage_hours": 5.6,
        "usage_pct": 86,
        "usage_window_days": 30,
        # applies to everyone
        "swo_on_file": True,
        # presentation only -- oracle() ignores both of these
        "denial_cites": None,
        "bait_quote": False,
    }
    spec.update(overrides)
    # reeval_late calls it adherence_pct; it is the same number as usage_pct.
    # spec["adherence_pct"] = spec["usage_pct"]
    return spec


def make_spec(bucket, i, rng):
    """bucket name -> a spec dict. One branch per bucket.

    Within a bucket the variants are picked by index rather than at random, so every
    intended edge case appears exactly once instead of being drawn twice by luck.
    """
    symptom = rng.choice(SYMPTOMS)

    if bucket == "clear_met":
        # two of the eight take the E0470 route, so the bi-level criteria are exercised
        # with a "met" somewhere in the set and not only with failures.
        # if i == 1:
        #     return new_spec(
        #         phase="continued",
        #         reeval_day=95,
        #     )
        if i % 4 == 0:
            return new_spec(device="E0470", e0601_tried=True, e0601_ineffective=True,
                            ahi=round(rng.uniform(18, 40), 1),
                            events=rng.randint(30, 180),
                            symptom=symptom)
        return new_spec(ahi=round(rng.uniform(18, 40), 1),
                        events=rng.randint(30, 180),
                        symptom=symptom)

    if bucket == "met_5_14":
        return new_spec(ahi=round(rng.uniform(7, 13), 1),
                        events=rng.randint(10, 60),
                        symptom=symptom)

    if bucket == "clear_unmet":
        # six different ways to clearly fail, rather than six copies of two ways. Each
        # failure is stated outright in the record -- these are unmet, never insufficient.
        variants = [
            {"ahi": round(rng.uniform(1, 4), 1), "events": rng.randint(4, 18),
             "symptom": "none"},                               # nowhere near either threshold
            {"ahi": 3.2, "events": 6, "study_hours": 1.2,      # short study, far too few events
             "symptom": "none"},
            {"device": "E0471"},                               # backup rate billed against OSA
            {"device": "E0471", "ahi": round(rng.uniform(16, 30), 1),
             "events": rng.randint(40, 90)},                   # qualifying study, wrong device
            {"initial_eval_before_test": False,                # the paperwork failures
             "instruction_given": False, "swo_on_file": False},
            {"device": "E0470", "e0601_tried": True,           # CPAP worked, so no bi-level
             "e0601_ineffective": False, "test_ordered_by": "other"},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    if bucket == "insufficient":
        variants = [
            {"ahi": None, "study_hours": None},                                    # no index anywhere in the record
            {"events": None},                                 # an index, but no event count
            {"ahi": round(rng.uniform(6, 13), 1),             # numbers land in the 5-14 route
             "events": rng.randint(12, 40), "symptom": None}, # but no symptom is ever stated
            {"instruction_given": None,                       # setup note is silent on both
             "initial_eval_before_test": None},
            {"swo_on_file": None, "medicare_valid": None},    # order and test validity unstated
            {"phase": "continued", "usage_hours": None,       # adherence never written down
             "usage_pct": None, "usage_window_days": None},
            {"phase": "continued", "reeval_day": None},       # follow-up undated
            {"phase": "continued", "symptoms_improved": None,
             "adherence_reviewed": None},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    if bucket == "borderline":
        variants = [
            {"ahi": 14.0, "events": rng.randint(20, 40)},      # top of the B2 range
            {"ahi": 15.0, "events": 25},                       # clears AHI, misses the event floor
            {"ahi": 5.0, "events": 10},                        # bottom of the B2 range, exactly
            {"ahi": 16.0, "events": 28, "study_hours": 1.5},   # short study, symptom carries it
            {"phase": "continued", "usage_hours": 4.0,         # four hours, but 68% of nights
             "usage_pct": 68, "adherence_reviewed": False},
            {"phase": "continued", "usage_hours": 3.9,         # enough nights, just under four hours
             "usage_pct": 88},
            {"phase": "continued", "reeval_day": 95, "symptoms_improved": False},          # past day 91, but documented
            {"device": "E0470", "e0601_tried": True,           # bi-level, no record CPAP failed
             "e0601_ineffective": None},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    # if bucket == "adversarial":
    #     variants = [
    #         {"denial_cites": "wrong_oxygen"},                  # denial argues the oxygen LCD
    #         # E0471 is only excluded when the PRIMARY diagnosis is OSA. Here it is not, so
    #         # the criterion is met and the denial is citing a rule that does not apply.
    #         {"device": "E0471",
    #          "denial_cites": "E0471_osa"},
    #         {"bait_quote": True, "ahi": round(rng.uniform(18, 30), 1)},
    #         {"bait_quote": True, "ahi": 9.0, "events": rng.randint(12, 30)},
    #     ]
    #     over = {"symptom": symptom}
    #     over.update(variants[i % len(variants)])
    #     return new_spec(**over)

    if bucket == "adversarial":
        variants = [
            {
                # The patient's late reevaluation is valid,
                # but the denial cites an unrelated oxygen policy.
                "phase": "continued",
                "reeval_day": 95,
                "denial_cites": "wrong_oxygen",
            },
            {
                # E0601 was requested, but the denial incorrectly
                # cites the E0471 prohibition.
                "device": "E0601",
                "denial_cites": "E0471_osa",
            },
            {
                # Incorrect policy-like wording is planted in
                # the patient record to tempt fabricated quoting.
                "bait_quote": True,
                "ahi": round(rng.uniform(18, 30), 1),
            },
            {
                "bait_quote": True,
                "ahi": 9.0,
                "events": rng.randint(12, 30),
            },
        ]

        over = {
            "symptom": symptom,
        }

        over.update(
            variants[i % len(variants)]
        )

        return new_spec(**over)


    raise ValueError(f"unknown bucket: {bucket}")

### Generate, and prove it is deterministic

The seed is fixed, so this must reproduce `data/cases.json` exactly. That matters for more than
tidiness: the case text goes into the prompt, so if a single word changed, every cached response
would miss and the experiment would cost real money to re-run.

In [ ]:
rng = random.Random(0)
cases, i = [], 0
for bucket, n in BUCKETS.items():
    for _ in range(n):
        i += 1
        spec = make_spec(bucket, i, rng)
        cases.append({"id": f"case_{i:03d}", "bucket": bucket, "spec": spec,
                      "documents": render(spec, rng), "gold": oracle(spec),
                      "hand_written": False})

on_disk = Path("data/cases.json")
if on_disk.exists():
    same = json.loads(on_disk.read_text()) == cases
    print("regenerated 40 cases; identical to data/cases.json:", same)
    assert same, "generator drifted -- the cached responses will all miss"
else:
    on_disk.write_text(json.dumps(cases, indent=2))
    print("wrote data/cases.json")

print("\nbuckets:", dict(Counter(c["bucket"] for c in cases)))
print("labels :", dict(Counter(l for c in cases for l in c["gold"].values())),
      f"over {sum(len(c['gold']) for c in cases)} decisions")

### What one case looks like

`case_024` is from the *insufficient* bucket: the sleep study never states an AHI. Read the sleep
study below and notice what is **not** there — there is no "AHI not recorded" line, just an absence.
That is what the model has to spot.

In [ ]:
demo = next(c for c in cases if c["id"] == "case_024")

for name, text in demo["documents"].items():
    print(f"----- {name} -----\n{text}\n")
print("gold labels:")
for cid, label in demo["gold"].items():
    print(f"  {cid:<22} {label}")

## 4. Retrieval — finding the right rule

Two chunking strategies, because the comparison is the experiment:

- **fixed** — cut every document into 512-word blocks. The dumb baseline.
- **criteria** — one chunk per rule for L33718, headings for everything else. A two-part rule
  never gets sliced in half.

Then three ways to search, each building on the last: dense embeddings, dense + BM25, and a
cross-encoder reranker on top.

In [ ]:
def chunk_fixed(text, doc_id, size=512, overlap=64):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(1, len(words)), step):
        w = words[i:i + size]
        if not w:
            break
        out.append({"id": f"{doc_id}::fix::{len(out)}", "doc": doc_id, "text": " ".join(w),
                    "criterion": None})
    return out

def chunk_headings(text, doc_id, min_chars=200):
    marks = list(re.finditer(r"^#{1,6}\s+(.*)$", text, re.M))
    if not marks:
        return chunk_fixed(text, doc_id)
    out = []
    for n, m in enumerate(marks):
        end = marks[n + 1].start() if n + 1 < len(marks) else len(text)
        body = text[m.start():end].strip()
        if len(body) < min_chars and out:
            out[-1]["text"] += "\n\n" + body
            continue
        out.append({"id": f"{doc_id}::sec::{len(out)}", "doc": doc_id, "text": body,
                    "criterion": None})
    return out

def chunk_by_criteria(criteria, policies):
    # text_core, not text: the raw quotes carry markdown list markers ("1. ")
    # that the model never reproduces. Indexing the clean sentence keeps the
    # normalized quotation check honest while keeping each rule intact.
    out = [{"id": f"crit::{c['id']}", "doc": c["source"],
            "text": c.get("text_core") or c["text"], "raw_text": c["text"],
            "criterion": c["id"],
            "phase": c["phase"], "device": c["device"]}
           for c in criteria if c["text"]]
    for doc_id, text in policies.items():
        if doc_id != "L33718":
            out.extend(chunk_headings(text, doc_id))
    return out

In [ ]:
fixed  = [c for d, t in policies.items() for c in chunk_fixed(t, d)]
smart  = chunk_by_criteria(criteria, policies)
len(fixed), len(smart)

# ---- from 03_pipeline: cell 10 ----

### Embeddings and BM25

About 200 chunks, so the "vector database" is one numpy array and a dot product. BM25 sits
alongside because embeddings are fuzzy about exact strings like `E0601` and BM25 is not. The two
scores are min-max normalised before averaging — they are on completely different scales.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # CPU is fine

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    M = embedder.encode(texts, normalize_embeddings=True)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([t.lower().split() for t in texts])
    return {"chunks": chunks, "M": np.asarray(M, dtype=np.float32), "bm25": bm25}

def _minmax(a):
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

def search(query, index, mode="dense", top_k=5, w=0.5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    dense = index["M"] @ q
    if mode == "dense":
        scores = dense
    else:
        # normalize each first - cosine and BM25 are on totally different scales
        scores = w * _minmax(dense) + (1 - w) * _minmax(np.asarray(index["bm25"].get_scores(query.lower().split())))
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in order]

# ---- from 03_pipeline: cell 12 ----

### The reranker

A cross-encoder reads the query and the passage *together* rather than embedding them separately,
which is slower but much sharper. It is loaded lazily, so the configs that do not use it never pay
the ~1.1GB download.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = None

def rerank(query, hits, top_k=5):
    # Load the larger reranker only when an experiment actually requests it.
    global reranker
    if reranker is None:
        reranker = CrossEncoder("BAAI/bge-reranker-base")
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(hits, scores), key=lambda x: -x[1])
    return [{**h, "score": float(s)} for h, s in ranked[:top_k]]

# ---- from 03_pipeline: cell 14 ----

### One query per rule, not one per case

This is the single change that mattered most in the whole project.

The first version built **one blended query per case** naming all six criteria at once and asked
for 12 passages covering all of them. The result: **18 of 18 criterion passages were never
retrieved.** Twenty short rule sentences cannot outrank 123 long decoy sections on a query that is
about six things simultaneously.

Searching once per rule and merging the results took target-rule coverage from **0 to 25/26**.
Everything downstream that looked like a model failure — invented quotes, collapsing accuracy —
was really this one bug, reported four different ways.

In [ ]:
from tqdm.auto import tqdm

CRITERIA_BY_ID = {c["id"]: c for c in criteria}

# Retrieval runs once per applicable rule, not once per case.
#
# The first smoke run sent a single blended query naming every criterion at once and
# asked for 12 passages covering all of them. 18 of 18 criterion passages came back
# unretrieved: twenty short rule sentences cannot outrank 123 long decoy sections on a
# query that is about six things simultaneously. Everything downstream -- made_up 0.78,
# row 5 collapsing to F1 0.07 -- was that one failure reported four times.
PER_CRITERION_K   = 3     # passages kept for each rule
RERANK_CANDIDATES = 24    # candidates the cross-encoder scores, per rule

def case_criteria(case):
    """Use the frozen oracle keys so conditional, irrelevant rules stay omitted."""
    return [CRITERIA_BY_ID[cid] for cid in case["gold"]]

def criterion_query(case, criterion):
    """One query per rule. Short and specific is what this retriever is good at."""
    return (
        f"Medicare coverage rule for device {case['spec']['device']} during the "
        f"{case['spec']['phase']} phase. {criterion['id']}: {criterion['summary']}"
    )


def whole_policy_hit():
    """Row 0's pseudo-passage: the entire LCD, no retrieval involved."""
    return {"id": "L33718::full", "doc": "L33718", "text": policies["L33718"],
            "criterion": None, "score": 1.0}


def retrieve_for_criterion(case, criterion, cfg, index):
    """The passages this ONE rule retrieves, in its own ranked order."""
    if cfg["retrieval"] is None:
        return [whole_policy_hit()]

    query = criterion_query(case, criterion)
    if cfg["retrieval"] == "rerank":
        candidates = search(query, index, mode="hybrid", top_k=RERANK_CANDIDATES)
        return rerank(query, candidates, top_k=PER_CRITERION_K)

    return search(query, index, mode=cfg["retrieval"], top_k=PER_CRITERION_K)


def retrieve_per_criterion(case, cfg, index):
    """-> ({criterion_id: its own ranked hits}, combined deduplicated context)

    Each rule is searched independently, then the results are merged into one context
    for a single Gemini call. Deduplication keeps first-seen order, and the criteria are
    walked in the order the oracle listed them, so the combined context is deterministic
    -- which is what lets row 5 reuse row 4's cached response.

    Ranks are always read from a rule's OWN list. A passage's position in the combined
    context is an artefact of merge order and says nothing about retrieval quality.
    """
    per_criterion, combined, seen = {}, [], set()
    for criterion in case_criteria(case):
        hits = retrieve_for_criterion(case, criterion, cfg, index)
        per_criterion[criterion["id"]] = hits
        for hit in hits:
            if hit["id"] not in seen:
                seen.add(hit["id"])
                combined.append(hit)
    return per_criterion, combined

### See it work

Retrieve for one rule and look at what comes back. `crit::adherence` should be rank 1 — and the
decoys should be visibly present in the index, otherwise the whole comparison is rigged.

In [ ]:
index_demo = build_index(smart)
crit_demo  = next(c for c in criteria if c["id"] == "adherence")
case_demo  = next(c for c in cases if c["id"] == "case_022")

q = criterion_query(case_demo, crit_demo)
print("query:", q, "\n")
for rank, hit in enumerate(search(q, index_demo, mode="hybrid", top_k=5), 1):
    mark = "  <-- the rule we wanted" if hit.get("criterion") == "adherence" else ""
    print(f"  {rank}. {hit['doc']:<12} {hit['id']:<28} {hit['score']:.3f}{mark}")

## 5. Asking the model

A Pydantic schema forces the JSON into a fixed shape, so a missing field fails loudly instead of
silently becoming `None` three cells later.

`ask()` caches every response on disk keyed by `model | system | schema | prompt`. On a free tier
that is the single most useful thing in the notebook: *Restart & Run All* costs nothing once the
cache is warm. The flip side is that **changing one character of the system prompt invalidates the
entire cache.**

In [ ]:
import hashlib, random, time
from typing import Literal

from pydantic import BaseModel, Field

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Stable, low-cost model suited to classification and structured extraction.
MODEL = "gemini-3.1-flash-lite"


class Decision(BaseModel):
    criterion_id: str
    label: Literal["met", "unmet", "insufficient_evidence"]
    evidence_quote: str = Field(
        description="Exact supporting quotation copied from the supplied policy text")
    source_doc_id: str
    reasoning: str
    model_reported_confidence: float = Field(
        ge=0.0, le=1.0,
        description=(
            "The model's own 0-1 confidence in this label. SELF-REPORTED AND "
            "UNCALIBRATED -- recorded for exploration only, never as a reliability "
            "estimate. The trustworthy signal in this project is quote_status, which "
            "is checked by string matching against the policy text."))


class DecisionBatch(BaseModel):
    decisions: list[Decision]


DECISION_SCHEMA = DecisionBatch.model_json_schema()


def ask(prompt, system="", model=MODEL, force=False, response_schema=None):
    """Ask the LLM, but only once per unique prompt. Repeats come off disk.

    This is the single most useful thing on a free tier. Restart & Run All costs
    zero API calls once the cache is warm.
    """
    schema_key = json.dumps(response_schema, sort_keys=True) if response_schema else ""
    key = hashlib.sha256(
        f"{model}|{system}|{schema_key}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    from google import genai   # SDK surface changes - check current docs if this errors
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            config = {
                "system_instruction": system,
                "temperature": 0,
                "response_mime_type": "application/json",
            }
            if response_schema is not None:
                config["response_json_schema"] = response_schema

            r = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config,
            )
            text = r.text
            break
        except Exception as e:
            message = str(e).lower()
            retryable = any(token in message for token in (
                "429", "503", "resource_exhausted", "unavailable"))
            if not retryable or attempt == 4:
                raise
            wait = 5 * (2 ** attempt) + random.random()
            print(f"temporary API error; retrying in {wait:.1f}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

def ask_json(prompt, system="", **kw):
    raw = ask(
        prompt, system, response_schema=DECISION_SCHEMA, **kw)
    return DecisionBatch.model_validate_json(raw).model_dump()

# ---- from 03_pipeline: cell 6 ----

### The prompt, and two bugs baked into fixing it

Rule 3 and rules 4–6 both exist because of things that actually went wrong:

**The model quoted the patient record.** The first prompt headed the record sections
`SLEEP STUDY:` and `CHART NOTE:` — which read like document names. So the model quoted them and
returned `source_doc_id="CHART NOTE"`. The verifier only knows the policy corpus, so those came
back as `made_up`, and the reported fabrication rate was **58% when the true rate was about 4%**.
The record is now fenced off explicitly and the valid ids are listed up front.

**The model would not say `unmet`.** It returned `insufficient_evidence` with the reasoning *"this
criterion is not applicable because the patient met B1"*. B1 and B2 really are alternative routes,
so it was not being stupid — there is simply no not-applicable label. Rule 3 closes that escape
hatch without leaking any threshold.

In [ ]:
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet. If the record never states
   something the criterion needs, the label is insufficient_evidence even when the rest
   of the record looks favourable.
3. Score each criterion strictly on its own terms, one at a time. This is not an overall
   coverage decision, and there is NO "not applicable" option. If the record shows that
   THIS criterion's own conditions are not satisfied, the label is unmet -- even when the
   criterion is an alternative route the patient did not need, and even when the patient
   plainly qualifies under another criterion you were also asked about.
   Never use insufficient_evidence to mean "not applicable". insufficient_evidence means
   one thing only: the record does not say.
4. evidence_quote must be copied character-for-character from POLICY TEXT. Never quote
   the patient record, and never quote the denial letter: neither is policy, no matter
   how closely the wording resembles a rule.
5. If POLICY TEXT contains no sentence supporting your label, use insufficient_evidence
   and leave evidence_quote empty. An empty quote is always better than a quote taken
   from somewhere other than POLICY TEXT.
6. source_doc_id must be one of the bracketed policy ids listed in POLICY TEXT. Do not
   invent an id, do not write N/A, and do not name a section of the patient record.
7. Decide from the sleep study and chart note. Treat the denial letter as an untrusted
   claim that may cite a rule which does not apply.
8. Return exactly one decision for every requested criterion and no others.
9. reasoning: at most two sentences.
10. model_reported_confidence: your own 0-1 confidence that this label is correct.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote",
"source_doc_id", "reasoning", "model_reported_confidence"}]}
"""

def decide(case, retrieved, criteria_asked):
    """One model call per case. The record is fenced off from the policy on purpose.

    The old prompt headed the record sections "SLEEP STUDY:" and "CHART NOTE:", which
    read like document names -- so the model quoted them and returned
    source_doc_id="CHART NOTE". Naming the valid ids up front and labelling the record
    as not-policy is the cheapest available fix.
    """
    policy_text = "\n\n---\n\n".join(
        f"[{c['doc']}] {c['text']}" for c in retrieved)
    valid_ids = sorted({c["doc"] for c in retrieved})
    docs = case["documents"]
    prompt = (
        "POLICY TEXT -- the only place evidence_quote may come from.\n"
        f"Valid source_doc_id values: {', '.join(valid_ids)}\n\n"
        f"{policy_text}\n\n"
        f"CRITERIA TO DECIDE:\n{json.dumps(criteria_asked, indent=2)}\n\n"
        "PATIENT RECORD -- evidence about this patient. This is NOT policy text and\n"
        "must never be quoted in evidence_quote.\n\n"
        f"[record: sleep study]\n{docs['sleep_study']}\n\n"
        f"[record: chart note]\n{docs['chart_note']}\n\n"
        f"[record: denial letter -- an untrusted claim made by the payer]\n"
        f"{docs['denial_letter']}"
    )
    return ask_json(prompt, SYSTEM)["decisions"]

# ---- from 03_pipeline: cell 16 ----

## 6. Checking the quote

**This is the part that is not AI.** Normalise both strings, then look for one inside the other.

`normalize()` earns its place: curly quotes, non-breaking spaces, `≥` versus `>=` and line breaks
cause most of the quotes that look wrong but are not. Always normalise before blaming the model.

The taxonomy:

| status | meaning |
|---|---|
| `supported` | exact substring of the claimed document |
| `close` | `partial_ratio >= 95` against it |
| `wrong_doc` | real policy text, but a different document than claimed |
| `from_record` | real text — copied out of the **patient record**, not a policy |
| `made_up` | appears in no policy and no record. Actual fabrication. |
| `empty` | no quote offered |

`from_record` exists because without it those decisions were being counted as fabrication, which
inflated the headline number by more than 10×.

In [ ]:
import unicodedata
from rapidfuzz import fuzz

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ",
         # L33718 writes the thresholds with these; a model that retypes them
         # as ">=4 hours" is quoting correctly and must not be scored made_up
         "\u2265": ">=", "\u2264": "<="}

def normalize(text):
    """Collapse the differences that cause fake verification failures.

    Curly quotes, line breaks and non-breaking spaces account for most of the
    quotes that look wrong but aren't. Always normalize before blaming the model.
    """
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

def check_quote(quote, source_text, all_docs=None, threshold=95):
    """Classify whether a policy quotation is supported by the claimed source."""
    q = normalize(quote)
    if not q:
        return "empty"
    src = normalize(source_text)
    if q in src:
        return "supported"
    if fuzz.partial_ratio(q, src) >= threshold:
        return "close"
    for other in (all_docs or {}).values():
        o = normalize(other)
        if q in o or fuzz.partial_ratio(q, o) >= threshold:
            return "wrong_doc"
    return "made_up"

# Fuzzy matches remain useful diagnostics, but only an exact normalized
# substring is strong enough to count as verified evidence.
VERIFIED = ("supported",)

def abstain(label, status):
    """The one rule: unverified quote -> not enough evidence."""
    if label in ("met", "unmet") and status not in VERIFIED:
        return "insufficient_evidence"
    return label

# ---- citation scope: a DIAGNOSTIC that never changes a label --------------------
# check_quote answers "is this sentence really in the document claimed?" -- provenance.
# It does NOT answer "does this sentence support the decision?". Nothing here can:
# string matching cannot read. This function is only the next question down, of the
# citations whose provenance checks out, how many quote the rule being decided.
#
# It is deliberately not wired into abstain(). Measured over the 40-case run, promoting
# it to an abstention trigger would have abstained on 6 decisions in the final config --
# all 6 of them correct, none of them wrong.

def citation_scope(quote, criterion, all_criteria):
    """-> 'own_rule' | 'other_rule' | 'policy_prose' | None

    None means there was no citation to place. Matching runs both ways, because the model
    may quote a fragment of the rule or a longer span that contains it.

    'other_rule' is rarer than it looks and needs care: criteria.json splits some
    continuous policy requirements into two rules, so a quote can land on a neighbour
    that says substantially the same thing.
    """
    q = normalize(quote)
    if not q:
        return None

    own = normalize(criterion.get("text_core") or criterion["text"])
    if own and (q in own or own in q):
        return "own_rule"

    for other in all_criteria:
        if other["id"] == criterion["id"]:
            continue
        t = normalize(other.get("text_core") or other["text"])
        if t and (q in t or t in q):
            return "other_rule"

    return "policy_prose"

In [ ]:
RECORD_SECTIONS = ("sleep_study", "chart_note", "denial_letter")


def quote_record_section(quote, case):
    """Which part of the patient record this quote was lifted from, if any."""
    q = normalize(quote)
    if not q:
        return None
    for section in RECORD_SECTIONS:
        if q in normalize(case["documents"][section]):
            return section
    return None


def classify_quote(quote, case, source_text, all_docs):
    """check_quote, plus the one thing check_quote cannot see.

    check_quote only knows the policy corpus, so a sentence lifted straight out of the
    chart note comes back "made_up" -- which reads as fabrication and is not. In the
    first smoke run 14 of 15 "made_up" decisions were verbatim patient-record text, and
    the reported fabrication rate was 58% when the true rate was about 4%.

    "from_record" names that error honestly: real text, wrong corpus. The verifier
    itself is untouched, and "from_record" is not in VERIFIED, so abstention behaves
    exactly as it did before. Only the label changes, not the pipeline.
    """
    status = check_quote(quote, source_text, all_docs)
    if status in VERIFIED:
        return status
    return "from_record" if quote_record_section(quote, case) else status

In [ ]:
def criterion_rank(criterion, hits, chunking):
    """1-based rank of the passage carrying this criterion; None means not retrieved.

    Criteria chunks are labelled with their criterion id, which is an exact signal.
    Matching on text instead would be wrong for them: the swo sentence is DME
    boilerplate that appears verbatim in seven of the eight decoy policies, so a
    text match would happily score a decoy chunk as a hit.

    Fixed-size chunks carry no label, so there text matching is all there is. Row 0
    puts the whole of L33718 in one pseudo-chunk, so every L33718 criterion scores
    rank 1 by construction -- that column is not meaningful for row 0.
    """
    if chunking == "criteria":
        for rank, hit in enumerate(hits, start=1):
            if hit.get("criterion") == criterion["id"]:
                return rank
        return None

    target = normalize(criterion.get("text_core") or criterion["text"])
    for rank, hit in enumerate(hits, start=1):
        if target and target in normalize(hit["text"]):
            return rank
    return None

### Prove the verifier actually works

No API calls — these run on text alone.

In [ ]:
_c = next(c for c in criteria if c["id"] == "apnea_def")
_case = cases[0]
_chart_line = _case["documents"]["chart_note"].splitlines()[1]

# real policy text verifies
assert classify_quote(_c["text_core"], _case, policies[_c["source"]], policies) == "supported"
# a line lifted from the patient record is from_record, not fabrication
assert classify_quote(_chart_line, _case, policies["L33718"], policies) == "from_record"
# invented text is caught
assert classify_quote("the moon is made of cheese", _case, policies["L33718"], policies) == "made_up"
# and an unverified quote can never support a decision
assert abstain("met", "made_up") == "insufficient_evidence"
assert abstain("unmet", "from_record") == "insufficient_evidence"
assert abstain("met", "supported") == "met"

print("verifier behaves as intended")
print("\nnormalisation in action:")
print("  policy writes :", repr("greater than or equal to 15"))
print("  model may type:", repr("\u2265 15"))
print("  normalised    :", normalize("\u2265 15"), "|", normalize("&gt;= 15".replace("&gt;", ">")))

## 7. One case, end to end

Everything above, on a single patient. This reproduces exactly what config `row4_rerank` does for
`case_001`, so **if the cache is present it costs nothing** — the prompt hash matches.

Set `RUN_LIVE_DEMO = False` to skip it entirely.

In [ ]:
def run_one_case(case, cfg, index):
    """Run one model request and return one result row per applicable criterion."""
    asked = case_criteria(case)
    per_criterion, combined = retrieve_per_criterion(case, cfg, index)
    criteria_asked = [
        {"id": c["id"], "summary": c["summary"]}
        for c in asked
    ]

    # Exactly one model call per case per config, on the merged context.
    decisions = decide(case, combined, criteria_asked)
    by_id = {d["criterion_id"]: d for d in decisions}
    expected_ids = {c["id"] for c in asked}
    extra_ids = sorted(set(by_id) - expected_ids)

    rows = []
    for criterion in asked:
        cid = criterion["id"]
        decision = by_id.get(cid)

        if decision is None:
            decision = {
                "criterion_id": cid,
                "label": "insufficient_evidence",
                "evidence_quote": "",
                "source_doc_id": "",
                "reasoning": "The model omitted this requested criterion.",
                "model_reported_confidence": None,
            }
            model_omitted = True
        else:
            model_omitted = False

        source_text = policies.get(decision["source_doc_id"], "")
        quote_status = classify_quote(
            decision["evidence_quote"], case, source_text, policies)
        record_section = quote_record_section(decision["evidence_quote"], case)
        # diagnostic only -- see the note beside citation_scope()
        scope = (citation_scope(decision["evidence_quote"], criterion, criteria)
                 if quote_status == "supported" else None)
        raw_label = decision["label"]
        final_label = (
            abstain(raw_label, quote_status)
            if cfg["verify"] and cfg["abstain"]
            else raw_label
        )

        rows.append({
            "config": cfg["name"],
            "case_id": case["id"],
            "bucket": case["bucket"],
            "hand_written": case["hand_written"],
            "device": case["spec"]["device"],
            "phase": case["spec"]["phase"],
            "criterion_id": cid,
            "gold": case["gold"][cid],
            "predicted_raw": raw_label,
            "predicted_final": final_label,
            "quote_status": quote_status,
            # which record section the quote came from, when it came from the record.
            # denial_letter is the worst case: the payer's paraphrase of a rule.
            "quote_record_section": record_section,
            "citation_scope": scope,
            "quote_matches_criterion": scope == "own_rule",
            "evidence_quote": decision["evidence_quote"],
            "source_doc_id": decision["source_doc_id"],
            "reasoning": decision["reasoning"],
            # self-reported and uncalibrated: see the Decision schema note
            "model_reported_confidence": decision.get("model_reported_confidence"),
            "model_omitted": model_omitted,
            "model_extra_ids": extra_ids,
            # measured in this criterion's own result list, not in the merged context
            "retrieval_rank": criterion_rank(
                criterion, per_criterion[cid], cfg["chunking"]),
            "criterion_retrieved_ids": [h["id"] for h in per_criterion[cid]],
            "retrieved_chunk_ids": [hit["id"] for hit in combined],
            "retrieved_doc_ids": [hit["doc"] for hit in combined],
        })

    return rows

In [ ]:
index_cache = {}


def get_index(cfg):
    if cfg["retrieval"] is None:
        return None

    key = cfg["chunking"]
    if key not in index_cache:
        chunks = smart if key == "criteria" else fixed
        print(f"building {key} index from {len(chunks)} chunks")
        index_cache[key] = build_index(chunks)
    return index_cache[key]


def run_config(cfg):
    index = get_index(cfg)
    rows = []
    output = RESULT_DIR / f"{cfg['name']}.json"

    for case in tqdm(selected_cases, desc=cfg["name"]):
        rows.extend(run_one_case(case, cfg, index))

        # Checkpoint after each case. Re-running is cheap because ask() uses its cache.
        output.write_text(json.dumps({
            "run": run_name,
            "model": MODEL,
            "config": cfg,
            "rows": rows,
        }, indent=2))

    return rows

In [ ]:
RUN_LIVE_DEMO = True      # one model call (free if the cache is warm)

cfg_demo = {"name": "row4_rerank", "chunking": "criteria",
            "retrieval": "rerank", "verify": False, "abstain": False}

if RUN_LIVE_DEMO:
    case = next(c for c in cases if c["id"] == "case_001")
    rows = run_one_case(case, cfg_demo, index_demo)

    print(f"{'criterion':<22} {'gold':<22} {'predicted':<22} {'quote':<12} rank")
    print("-" * 88)
    for r in rows:
        mark = "" if r["gold"] == r["predicted_final"] else "   <-- disagrees with the oracle"
        print(f"{r['criterion_id']:<22} {r['gold']:<22} {r['predicted_final']:<22} "
              f"{r['quote_status']:<12} {r['retrieval_rank']}{mark}")

    print("\nthe quote it used for B1:")
    b1 = next(r for r in rows if r["criterion_id"] == "B1")
    print(" ", b1["evidence_quote"][:160])
    print("  reasoning:", b1["reasoning"])
else:
    print("skipped")

## 8. The experiment

Six configurations. Each differs from the one above it by **exactly one setting** — that is what
makes it an experiment rather than six unrelated runs.

Row 0 is required and cannot be skipped: L33718 fits in a modern context window, so the obvious
question is "why build retrieval at all?" Run it and publish the answer, even if it wins.

In [ ]:
CONFIGS = [
    {"name": "row0_context_only", "chunking": None,       "retrieval": None,     "verify": True,  "abstain": False},
    {"name": "row1_naive",        "chunking": "fixed",    "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row2_structure",    "chunking": "criteria", "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row3_hybrid",       "chunking": "criteria", "retrieval": "hybrid", "verify": False, "abstain": False},
    {"name": "row4_rerank",       "chunking": "criteria", "retrieval": "rerank", "verify": False, "abstain": False},
    {"name": "row5_full",         "chunking": "criteria", "retrieval": "rerank", "verify": True,  "abstain": True},
]

### Preflight — free, and it can save you 200 API calls

Retrieval only, no model calls. If target-rule coverage is near zero here, every number downstream
is a retrieval artefact and running the experiment would only spend quota proving it. That is
exactly what the first run did.

In [ ]:
RUN_EXPERIMENT = False    # True re-runs all six configs (~200 calls, free if cache is warm)
RUN_NAME       = "full"   # "full" = all 40 cases, "smoke" = the 4-case subset

SMOKE_CASE_IDS = {"case_001", "case_022", "case_031", "case_037"}
selected_cases = (cases if RUN_NAME == "full"
                  else [c for c in cases if c["id"] in SMOKE_CASE_IDS])
RESULT_DIR = Path("data/results") / RUN_NAME
RESULT_DIR.mkdir(parents=True, exist_ok=True)
run_name = RUN_NAME

print(f"{RUN_NAME}: {len(selected_cases)} cases, "
      f"{sum(len(c['gold']) for c in selected_cases)} decisions per config")

In [ ]:
# Retrieval-only preflight. Builds the indexes and measures how often each rule's own
# passage is actually retrieved. NO Gemini calls happen here.
#
# Read this before running the experiment. If target-rule coverage is near zero, every
# number downstream is a retrieval artefact and running the configs would only spend
# quota proving it. That is exactly what the first smoke run did.

print(f"{'config':<20} {'target rule found':>18} {'mean rank':>10} {'ctx chunks':>11}")
print("-" * 62)
for cfg in CONFIGS:
    if cfg["retrieval"] is None:
        print(f"{cfg['name']:<20} {'n/a whole policy':>18} {'-':>10} {1:>11}")
        continue

    index = get_index(cfg)
    found = total = 0
    ranks, sizes = [], []
    for case in selected_cases:
        per_criterion, combined = retrieve_per_criterion(case, cfg, index)
        sizes.append(len(combined))
        for criterion in case_criteria(case):
            rank = criterion_rank(criterion, per_criterion[criterion["id"]],
                                  cfg["chunking"])
            total += 1
            if rank is not None:
                found += 1
                ranks.append(rank)

    coverage = f"{found}/{total}"
    mean_rank = f"{sum(ranks) / len(ranks):.2f}" if ranks else "-"
    print(f"{cfg['name']:<20} {coverage:>18} {mean_rank:>10} "
          f"{sum(sizes) / len(sizes):>11.1f}")

print("\nrank is within each rule's own results, so 1.00 is perfect and "
      f"{PER_CRITERION_K} is the worst retrievable rank.")

Reading that table: **rank is measured inside each rule's own results**, so 1.00 is perfect
and `PER_CRITERION_K` is the worst retrievable rank. A passage's position in the merged context
says nothing about how well it was retrieved — that distinction is why `criterion_rank` takes the
per-criterion list rather than the combined one.

In [ ]:
if RUN_EXPERIMENT:
    all_rows = {}
    for cfg in CONFIGS:
        rows = run_config(cfg)
        all_rows[cfg["name"]] = rows
        correct = sum(r["predicted_final"] == r["gold"] for r in rows)
        print(f"{cfg['name']:<20} decisions={len(rows):<4} correct={correct}")
else:
    print(f"not running -- will read the results already in {RESULT_DIR}")

## 9. The numbers

Every figure gets a 95% interval. With 40 cases, "88%" is misleading — it sounds precise and it is
not.

Two different intervals for two different things: **Wilson** for proportions, because it behaves at
small n where the textbook normal approximation does not; and a **bootstrap that resamples the 40
cases** for F1. Not the 239 decisions — every decision for one patient comes from the same
retrieved context and the same single model call, so resampling decisions would treat correlated
observations as independent and report intervals far tighter than the evidence supports.

In [ ]:
import matplotlib.pyplot as plt

runs = {p.stem: json.load(open(p)) for p in sorted(RESULT_DIR.glob("*.json"))}
if not runs:
    raise FileNotFoundError(f"no results in {RESULT_DIR} -- set RUN_EXPERIMENT = True")

ORDER = ["row0_context_only", "row1_naive", "row2_structure",
         "row3_hybrid", "row4_rerank", "row5_full"]
AVAILABLE = [n for n in ORDER if n in runs]
MAIN = "row5_full" if "row5_full" in runs else AVAILABLE[-1]

for name in AVAILABLE:
    rws = runs[name]["rows"]
    print(f"  {name:<20} {len(rws):>5} decisions   "
          f"{len({r['case_id'] for r in rws}):>2} cases")

In [ ]:
import math

def wilson(successes, n, z=1.96):
    """95% interval for a proportion."""
    if n == 0:
        return (float("nan"),) * 3
    p = successes / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return p, max(0, centre - half), min(1, centre + half)

def bootstrap(items, stat, n_resamples=1000, seed=0, cluster="case_id"):
    """Resample CASES, not individual decisions.

    Every decision for one patient comes from the same retrieved context and the same
    single model call, so decisions within a case are correlated. Resampling decisions
    would treat ~239 correlated observations as 239 independent ones and report intervals
    that are far too narrow -- the classic way to make a small study look precise.
    PLAN.md specifies resampling the 40 cases, so that is what this does.

    Expect wider intervals than a naive bootstrap. That is the point.
    """
    if not items:
        return (float("nan"),) * 3
    rng = np.random.default_rng(seed)

    by_case = {}
    for r in items:
        by_case.setdefault(r[cluster], []).append(r)
    keys = list(by_case)

    draws = []
    for _ in range(n_resamples):
        picked = rng.integers(0, len(keys), len(keys))
        sample = [r for i in picked for r in by_case[keys[i]]]
        draws.append(stat(sample))

    lo, hi = np.percentile(draws, [2.5, 97.5])
    return stat(items), float(lo), float(hi)

def fmt(point, lo, hi):
    if any(math.isnan(v) for v in (point, lo, hi)):
        return "—"
    return f"{point:.2f} [{lo:.2f}, {hi:.2f}]"

print(fmt(*wilson(35, 40)))

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

LABELS  = ["met", "unmet", "insufficient_evidence"]
ABSTAIN = "insufficient_evidence"

def macro_f1(rows, field="predicted_final"):
    return f1_score([r["gold"] for r in rows], [r[field] for r in rows],
                    labels=LABELS, average="macro", zero_division=0)

def summarise(run):
    rows = run["rows"]
    n    = len(rows)
    cfg  = run["config"]

    # Row 0 puts the whole of L33718 into one pseudo-chunk, so every criterion "ranks"
    # first by construction. That is not a retrieval result, so it is not reported.
    if cfg["retrieval"] is None:
        found = "—"
    else:
        found = fmt(*wilson(sum(r["retrieval_rank"] is not None for r in rows), n))

    answered = [r for r in rows if r["predicted_final"] != ABSTAIN]
    return [
        found,
        fmt(*bootstrap(rows, macro_f1)),
        fmt(*wilson(sum(r["quote_status"] == "supported"   for r in rows), n)),
        # from_record and made_up are different failures and must not be added together.
        # from_record = real text copied out of the patient record, wrong corpus.
        # made_up     = text that appears in no policy and no record. Actual fabrication.
        fmt(*wilson(sum(r["quote_status"] == "from_record" for r in rows), n)),
        fmt(*wilson(sum(r["quote_status"] == "made_up"     for r in rows), n)),
        fmt(*wilson(sum(r["predicted_final"] == ABSTAIN  for r in rows), n)),
        fmt(*wilson(sum(r["gold"] == r["predicted_final"] for r in answered), len(answered))),
    ]

In [ ]:
HEAD  = ["config", "found rule", "F1", "quotes real", "from record", "made up",
         "abstained", "acc. answered"]
WIDTH = [20, 19, 19, 19, 19, 19, 19, 19]

print(" | ".join(h.ljust(w) for h, w in zip(HEAD, WIDTH)))
print("-" * (sum(WIDTH) + 3 * len(WIDTH)))
for name in AVAILABLE:
    cells = [name] + summarise(runs[name])
    print(" | ".join(str(c).ljust(w) for c, w in zip(cells, WIDTH)))

print()
print("found rule: row 1 chunks by size, so its column asks whether the rule's sentence")
print("  appears anywhere inside a retrieved 512-word block. Rows 2-5 require the exact")
print("  criterion chunk. Row 1's number is the easier test and is not directly")
print("  comparable with the rows below it.")

# then paste this into README.md

### Reading that table honestly

**What it supports:** retrieval improves monotonically, 0.34 → 0.65 → 0.67 → 0.97, and those
intervals do not overlap. That is a real result.

**What it does not support:** the F1 column cannot rank the configurations. Every interval overlaps
every other one, and row 0 — no retrieval at all — sits inside all of them. With 40 cases that
column is not precise enough to order anything.

One column name needs explaining: **"abstained" means the model predicted
`insufficient_evidence`**, which here is a real label with its own gold answers, not a refusal.

In [ ]:
print(f"{'config':<20} {'raw F1':>8} {'final F1':>9} {'forced abstentions':>20}")
for name in AVAILABLE:
    rows = runs[name]["rows"]
    forced = sum(r["predicted_raw"] != ABSTAIN and r["predicted_final"] == ABSTAIN
                 for r in rows)
    print(f"{name:<20} {macro_f1(rows, 'predicted_raw'):>8.2f} "
          f"{macro_f1(rows, 'predicted_final'):>9.2f} {forced:>20}")

### Did abstention do anything? No.

Zero forced abstentions in every config. Once retrieval and the prompt were fixed, the quote check
had nothing left to catch, so row 5 is identical to row 4 on every metric.

That is worth stating plainly rather than hiding. The abstention machinery earned its place against
an earlier, broken version of this pipeline — not against this one.

### Where the errors actually land

In [ ]:
rows = runs[MAIN]["rows"]
cm = confusion_matrix([r["gold"] for r in rows],
                      [r["predicted_final"] for r in rows], labels=LABELS)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_xticks(range(3), LABELS, rotation=45, ha="right")
ax.set_yticks(range(3), LABELS)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title(MAIN)
plt.tight_layout()

### Risk-coverage

Sweep a threshold and plot how much it answers against how wrong it is on what it answered. Two
signals could order the decisions and they are **not** equally trustworthy: `quote_status` is
objective, produced by string matching; the model's confidence is generated text. The dashed line
is plotted only so you can see it carries no signal.

In [ ]:
# higher = more trustworthy
# Ranked by how much the quote actually supports the decision. wrong_doc still cited
# real policy language and only mis-attributed it; from_record cited the patient's own
# paperwork as though it were a rule, which is further from evidence of coverage.
VERIFICATION_ORDER = {"supported": 5, "close": 4, "wrong_doc": 3,
                      "from_record": 2, "made_up": 1, "empty": 0}

def risk_coverage(rows, key):
    scored  = sorted(rows, key=key, reverse=True)
    correct = np.array([r["gold"] == r["predicted_raw"] for r in scored], dtype=float)
    k       = np.arange(1, len(scored) + 1)
    return k / len(scored), 1 - np.cumsum(correct) / k

rows = runs[MAIN]["rows"]
fig, ax = plt.subplots(figsize=(6, 4))

cov, err = risk_coverage(rows, lambda r: VERIFICATION_ORDER.get(r["quote_status"], 0))
ax.plot(cov, err, label="quote verification (objective)", linewidth=2)

# Exploratory only -- see the note above. Absent if the run predates the schema change.
conf = [r.get("model_reported_confidence") for r in rows]
if any(c is not None for c in conf):
    cov2, err2 = risk_coverage(rows, lambda r: (r.get("model_reported_confidence") or 0.0))
    ax.plot(cov2, err2, "--", label="model self-reported confidence (uncalibrated)")

ax.set_xlabel("coverage (fraction answered)")
ax.set_ylabel("error rate on answered")
ax.set_title(f"risk–coverage, {MAIN}")
ax.legend()
plt.tight_layout()

In [ ]:
rows = runs[MAIN]["rows"]          # set explicitly: do not inherit from an earlier cell

conf_rows = [r for r in rows if r.get("model_reported_confidence") is not None]

if not conf_rows:
    print("no model_reported_confidence in this run")
else:
    vals = [r["model_reported_confidence"] for r in conf_rows]
    print(f"confidence over {len(vals)} decisions: "
          f"min {min(vals):.2f}  max {max(vals):.2f}  mean {sum(vals)/len(vals):.3f}")

    right = [r["model_reported_confidence"] for r in conf_rows
             if r["gold"] == r["predicted_raw"]]
    wrong = [r["model_reported_confidence"] for r in conf_rows
             if r["gold"] != r["predicted_raw"]]

    if right and wrong:
        mr, mw = sum(right) / len(right), sum(wrong) / len(wrong)
        print(f"  when correct (n={len(right):>3}): {mr:.3f}")
        print(f"  when wrong   (n={len(wrong):>3}): {mw:.3f}")
        print(f"  gap                  : {mr - mw:+.3f}")
        if abs(mr - mw) < 0.05:
            print("  -> the model is about as confident when wrong as when right.")
            print("     It carries no usable signal. Report it as a negative result;")
            print("     do not use it to rank, threshold or abstain.")
    else:
        print("  need both correct and incorrect decisions to compare")

    # The objective signal, measured the same way, across EVERY config -- MAIN alone may
    # have no failed quotes left, which hides the interesting rows.
    print("\naccuracy by quote_status (objective, not self-reported):")
    print(f"  {'config':<20} " + " ".join(f"{s:>16}" for s in
          ("supported", "made_up", "empty")))
    for name in AVAILABLE:
        rws = runs[name]["rows"]
        cells_ = []
        for status in ("supported", "made_up", "empty"):
            sub = [r for r in rws if r["quote_status"] == status]
            cells_.append(f"{sum(r['gold'] == r['predicted_raw'] for r in sub) / len(sub):.2f}"
                          f" (n={len(sub)})" if sub else "-")
        print(f"  {name:<20} " + " ".join(f"{c:>16}" for c in cells_))
    print("\n  A config that is accurate on rows whose quote could NOT be verified is")
    print("  reaching the right answer with an invented citation. Abstaining on those")
    print("  rows buys trustworthy citations at the cost of accuracy -- that trade is")
    print("  the whole argument, and it should be stated as a trade, not a free win.")
    print("\nIf accuracy separates across these rows but not across confidence, that is")
    print("the argument for verifying quotes rather than asking the model how sure it is.")

### The finding that reverses the original thesis

Look at the `made_up` column above. Across all six configs, **23 decisions cited a quote that
appears in no policy and no patient record — and all 23 had the correct label.** Decisions where
the model honestly returned an empty quote were only about 69% correct.

So quote verification guarantees every citation shown to a human is real. It does **not** identify
wrong answers. On this data, abstaining on unverified quotes would have thrown away 23 correct
decisions to remove none that were wrong.

The model's self-reported confidence is no better: mean 0.995, and the gap between its confidence
when right and when wrong is about **+0.01**. It is recorded for exploration and should never be
used to rank, threshold or abstain.

In [ ]:
rows = runs[MAIN]["rows"]

print("quote status, every config:")
for name in AVAILABLE:
    c = Counter(r["quote_status"] for r in runs[name]["rows"])
    total = sum(c.values())
    parts = "  ".join(f"{k}={c[k]}" for k in
                      ("supported", "close", "wrong_doc", "from_record", "made_up", "empty")
                      if c[k])
    print(f"  {name:<20} {parts}")

sections = Counter(r.get("quote_record_section") for r in rows
                   if r.get("quote_record_section"))
if sections:
    print("\nwhen the model quoted the record instead of the policy, it quoted:")
    for sec, k in sections.most_common():
        note = "   <- the payer's own paraphrase of a rule" if sec == "denial_letter" else ""
        print(f"  {sec:<16} {k:>4}{note}")

print()
omitted = sum(r["model_omitted"] for r in rows)
print(f"criteria the model was asked for but did not return: {omitted}/{len(rows)}")

missed = sum(r["retrieval_rank"] is None for r in rows)
print(f"criterion passage never retrieved:                   {missed}/{len(rows)}")
print(f"distinct chunks retrieved across all cases:          "
      f"{len({c for r in rows for c in r['retrieved_chunk_ids']})}")

# Three kinds of document, not two. Calling everything-but-L33718 a decoy was wrong:
# A52467 is the PAP policy article and the two NCDs are the national CPAP and sleep-test
# rules, so retrieving them is reasonable. Only the eight unrelated equipment LCDs are
# there to be wrong answers, and only those tell you whether retrieval is being fooled.
TARGET     = "L33718"
SUPPORTING = {"A52467", "A55426", "NCD240.4", "NCD240.4.1"}

def doc_class(doc):
    if doc == TARGET:
        return "target"
    return "supporting" if doc in SUPPORTING else "decoy"

counts = Counter(d for r in rows for d in r["retrieved_doc_ids"])
total  = sum(counts.values())

print("\nretrieved documents:")
for doc, k in counts.most_common():
    print(f"  {doc:<12} {k:>5}   {doc_class(doc)}")

by_class = Counter()
for doc, k in counts.items():
    by_class[doc_class(doc)] += k

print("\n  share of retrieved context:")
for cls in ("target", "supporting", "decoy"):
    share = by_class[cls] / total if total else 0
    print(f"    {cls:<12} {by_class[cls]:>5}   {share:.0%}")
print("\n  decoys are the eight unrelated equipment LCDs; a high decoy share means")
print("  retrieval is being pulled away from the policy that actually governs the case.")

# macro-F1 averages over all three labels, so inside a bucket that contains only two of
# them it is capped at 2/3 -- which looks like a bad score and is not one. Report accuracy
# per bucket and show the cap alongside, so 0.67 is never mistaken for a finding.
print("\nby bucket:")
print(f"  {'bucket':<14} {'n':>4} {'classes':>8} {'accuracy':>9} {'macro-F1':>9}  {'':<12}")
for b in sorted({r["bucket"] for r in rows}):
    sub = [r for r in rows if r["bucket"] == b]
    classes = {r["gold"] for r in sub}
    acc = sum(r["gold"] == r["predicted_final"] for r in sub) / len(sub)
    cap = len(classes) / len(LABELS)
    note = "" if len(classes) == len(LABELS) else f"  <- F1 capped at {cap:.2f}"
    print(f"  {b:<14} {len(sub):>4} {len(classes):>8} {acc:>9.2f} {macro_f1(sub):>9.2f}{note}")

print("\naccuracy by criterion:")
for cid in sorted({r["criterion_id"] for r in rows}):
    sub = [r for r in rows if r["criterion_id"] == cid]
    ok  = sum(r["gold"] == r["predicted_final"] for r in sub)
    print(f"  {cid:<22} {ok}/{len(sub)}")

In [ ]:
def alt_route_rows(rows):
    """Decisions where gold=unmet only because the patient took the OTHER route."""
    gold_by_case = {}
    for r in rows:
        gold_by_case.setdefault(r["case_id"], {})[r["criterion_id"]] = r["gold"]

    flagged = set()
    for i, r in enumerate(rows):
        g = gold_by_case[r["case_id"]]
        other = {"B1": "B2", "B2": "B1"}.get(r["criterion_id"])
        if other and r["gold"] == "unmet" and g.get(other) == "met":
            flagged.add(i)
    return flagged


print(f"{'config':<20} {'primary F1':>11} {'excl. alt-route':>16} {'n excluded':>11}")
print("-" * 62)
for name in AVAILABLE:
    rws = runs[name]["rows"]
    flagged = alt_route_rows(rws)
    kept = [r for i, r in enumerate(rws) if i not in flagged]
    primary = macro_f1(rws)
    excl = macro_f1(kept) if kept else float("nan")
    # If excluding these empties a whole gold class, macro-F1 scores that class 0 and the
    # two columns stop being comparable. On the 4-case smoke set every unmet gold IS an
    # alternative-route decision, so the right-hand column is capped at 0.67 by
    # construction. With 40 cases there are 25 unmet golds that are not alternative-route,
    # and the comparison becomes meaningful.
    lost = set(r["gold"] for r in rws) - set(r["gold"] for r in kept)
    flag = f"   <- excluding these removed the '{'/'.join(sorted(lost))}' class" if lost else ""
    print(f"{name:<20} {primary:>11.2f} {excl:>16.2f} {len(flagged):>11}{flag}")

# how the model actually labelled those decisions
rws = runs[MAIN]["rows"]
flagged = alt_route_rows(rws)
if flagged:
    print(f"\nwhat the model said on the {len(flagged)} alternative-route decisions ({MAIN}):")
    for lab, k in Counter(rws[i]["predicted_raw"] for i in flagged).most_common():
        print(f"  {lab:<24} {k}")
    print("\n  a sample of its reasoning:")
    for i in sorted(flagged)[:3]:
        print(f"    {rws[i]['case_id']} {rws[i]['criterion_id']}: {rws[i]['reasoning'][:100]}")
else:
    print("\nno alternative-route decisions in this run")

print("\nRemaining errors after excluding them, by criterion:")
kept = [r for i, r in enumerate(rws) if i not in flagged]
wrong = Counter(r["criterion_id"] for r in kept if r["gold"] != r["predicted_final"])
print("  none" if not wrong else
      "\n".join(f"  {c:<22} {k}" for c, k in wrong.most_common()))

## Citation scope — does the quote cite the rule being decided?

`quotes real` in the table above is **provenance**: the sentence really does appear in the
document the model named. That is all a string match can establish. **It does not show that the
sentence logically supports the label** — nothing in this project checks that, and no amount of
string matching could.

This is the next question down. Of the citations whose provenance checks out, how many quote the
rule actually being decided?

- **own_rule** — the quote is inside that criterion's canonical sentence, or contains it
- **other_rule** — it matches a *different* criterion's sentence
- **policy_prose** — real policy text, but not one of the 20 curated sentences

`policy_prose` is not an error. `criteria.json` stores one canonical sentence per rule, not every
sentence in L33718 that supports it, so quoting a different supporting sentence lands here.

`other_rule` needs care too: some continuous policy requirements were split into two criteria, so
a quote can land on a neighbour that says substantially the same thing.

**This is a diagnostic and does not trigger abstention.** Measured over this run, promoting it to
an abstention trigger would have abstained on 6 decisions in the final config — all 6 correct,
none wrong.

In [ ]:
import unicodedata

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ", "\u2265": ">=", "\u2264": "<="}

def _norm(text):
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

CRITERIA = json.load(open("data/criteria.json"))
_CORE = {c["id"]: _norm(c.get("text_core") or c["text"]) for c in CRITERIA}

def scope_of(row):
    """Recomputed here rather than read from the row, so this works on result files
    written before notebook 04 started recording it."""
    if row["quote_status"] != "supported":
        return None
    q = _norm(row["evidence_quote"])
    if not q:
        return None
    own = _CORE.get(row["criterion_id"], "")
    if own and (q in own or own in q):
        return "own_rule"
    for cid, text in _CORE.items():
        if cid != row["criterion_id"] and text and (q in text or text in q):
            return "other_rule"
    return "policy_prose"


print(f"{'config':<20} {'own':>5} {'other':>6} {'prose':>6} | "
      f"{'own / citations':>16} {'own / all decisions':>21}")
print("-" * 82)
for name in AVAILABLE:
    rws   = runs[name]["rows"]
    scope = Counter(scope_of(r) for r in rws)
    offered = sum(bool(r["evidence_quote"].strip()) for r in rws)   # non-empty citations
    own     = scope["own_rule"]
    print(f"{name:<20} {own:>5} {scope['other_rule']:>6} {scope['policy_prose']:>6} | "
          f"{own:>4}/{offered:<4} = {own/offered:>5.1%}   "
          f"{own:>4}/{len(rws):<4} = {own/len(rws):>5.1%}")

print("\n  own / citations      denominator = decisions where a quote was offered at all")
print("  own / all decisions  denominator = every decision, empty citations included")
print("\n  Both are reported because they answer different questions: the first is about")
print("  citation quality when the model cites, the second folds in how often it declines.")

rws = runs[MAIN]["rows"]
off = [r for r in rws if scope_of(r) == "other_rule"]
if off:
    print(f"\nthe {len(off)} other_rule citations in {MAIN}, by criterion:")
    for cid, k in Counter(r["criterion_id"] for r in off).most_common():
        print(f"  {cid:<22} {k}")
    print("\n  one of them, in full:")
    print("   decided:", off[0]["criterion_id"])
    print("   quoted :", off[0]["evidence_quote"][:150])

## 10. What this does and does not show

**Real results**

- Chunking one rule per chunk, plus a cross-encoder reranker, takes target-rule retrieval from
  0.34 to 0.97 with non-overlapping intervals.
- Per-criterion accuracy is near ceiling on unambiguous rules; the hardest bucket is
  *insufficient* (0.87) — cases where the record is silent, which is the distinction that matters.
- BM25 measurably **hurts** here (row 3 has the worst F1 in the table).

**Honest limits**

- Labels are mine, not a clinician's.
- **Every case is templated.** Each field is phrased from a bank of three or four variants, so the
  model could be matching phrasing rather than reading. PLAN.md called for rewriting ten cases by
  hand as a control and that was never done — so treat the accuracy figures as an upper bound.
- n = 40, one policy, one model, one revision.
- Criterion selection is **given, not predicted**. The pipeline is told which rules apply, using
  the same phase and device gates the oracle uses. Gold *labels* never enter the prompt. This
  measures adjudication, not triage.

**If I picked this up again**, in order: write the ten handwritten cases and see whether the
numbers hold; work out why BM25 hurts; and only then look at entailment checking, which PLAN.md
rightly lists as the thing to add *after* everything else is finished.